# Stage 2 Notebook 47 - Exp2RR DETR K=64 queries + Hungarian + VFL

**Architectural escape from the 192-anchor design.** Across NB39-45 the per-prior ROI feature could not discriminate among 192 anchors that all sample similar lane-y regions. The cls collapse was not a loss issue (NB39-42), not a matching issue (NB44), not a capacity issue (NB45), not a representation issue (NB43). It might be intrinsic to the geometric-prior design itself.

Exp2RR abandons geometric priors and uses DETR-style learned queries instead. K=64 (vs LaneQueryHead default K=12 used in NB22-26) is large enough to cover the dataset's max ~10 lanes per image with comfortable slack for Hungarian 1-to-1 matching. Each query is a learned vector with no preset geometric bias -- queries differentiate via cross-attention to the multi-scale feature map. With Hungarian 1-to-1, each query gets at most ONE GT lane in any image, so the cls labels are stable per query across batches.

Combined with the new VFL recipe from Exp2QQ, this tests the orthogonal architectural angle.

Config:
- `lane_head.type: clrkd -> query`
- `lane_head.num_queries: 64`
- `lane_assigner: dynamic_k -> hungarian`
- `cls_loss_type: asl -> vfl` (same recipe as Exp2QQ)
- All else = NB40.

### Run mode

1. `DEBUG_MODE = True` smoke first -- larger query head + new loss path.
2. `DEBUG_MODE = False` for 20 epochs short run.
3. Wall-clock ~ 30 min on Colab Pro+.
4. Independent of NB35-46.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp42_rmt_gca_query64_hungarian_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp42_rmt_gca_query64_hungarian_vfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp42_rmt_gca_query64_hungarian_vfl_joint_smoke.log
OK exp42_rmt_gca_query64_hungarian_vfl_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.1400 det_loss=2.8631 grad_cos=0.0025 lambda_lane=0.1334
  gate_stats={'gate/det_mean': 0.5005649328231812, 'gate/lane_mean': 0.500846266746521, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp42_rmt_gca_query64_hungarian_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp42_rmt_gca_query64_hungarian_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp42_rmt_gca_query64_hungarian_vfl_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp42_rmt_gca_query64_hungarian_vfl_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp42_rmt_gca_query64_hungarian_vfl_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp42_rmt_gca_query64_hungarian_vfl_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp42_rmt_gca_query64_hungarian_vfl_j

0

## What to watch in Exp2RR training

Pass criteria at epoch 20:
- **`val/lane/decoded_f1 >= 0.15`**.
- **`pos_score - neg_score >= 0.20`** -- DETR queries with 1-to-1 Hungarian and VFL should produce a very clean cls separation since each query has a deterministic single-GT label or a stable no-object label.
- **`val/matched_line_iou >= 0.30`** -- query head's geometry is typically lower than anchor head, but we want at least the level of NB22 (matched_iou = 0.42 with K=12 queries) -- ideally higher with K=64.
- **`val/lane/decoded_oracle_f1 >= 0.30`**.

Failure signals:
- matched_iou < 0.20: K=64 queries can't ground themselves on the dataset (too many no-object slots, weak signal). Try `num_queries=32`.
- decoded_f1 ~ pos-neg gap small: cls couldn't separate even with the cleanest matching. Confirms the 192-anchor cls collapse is per-prior-feature-bound and we need richer features (Exp2QQ + bigger backbone OR pre-trained init).